In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *
from pyspark.sql.window import *

### Scenario:
You're given `app_logs_multiline.txt` — raw application logs where a **single logical record can span multiple physical lines**. Every genuine new log entry starts with a timestamp (`yyyy-MM-dd HH:mm:ss LEVEL message`). Lines that *don't* start with a timestamp (like stack trace frames or wrapped error details) are **continuations** of the previous record, not new records. Naively reading this line-by-line would fragment every multi-line error into several meaningless separate rows — you need to stitch continuation lines back onto their parent record. This is a real, common headache with legacy application/server logs.

**Problem:**

- Read the file as plain text (one row per physical line).
- Using `rlike()`, flag each line as either the **start of a new record** (matches pattern `^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}`) or a **continuation line** (doesn't match).
- Preserve the original file order (use `monotonically_increasing_id()` right after reading, before any other transformation, so you have a stable ordering column).
- Using a cumulative sum of the "new record" flag (ordered by that stable id), assign a `record_id` grouping each continuation line with the timestamped line above it.
- Group by `record_id`, and for each group:
  - Extract `log_date`, `log_time`, and `log_level` from the **first line** of that group (the timestamped one).
  - Concatenate all lines in the group (the main message plus any continuation lines) into one `full_message` string, joined with `" | "` — continuation lines should have their leading whitespace trimmed first.
- Order the final output by `log_date`, `log_time` ascending.

**Expected Output**

| log_date | log_time | log_level | full_message |
| :--- | :--- | :--- | :--- |
| 2024-11-01 | 10:00:01 | INFO | Service started successfully |
| 2024-11-01 | 10:00:05 | ERROR | Failed to connect to database \| Connection timeout after 30000ms \| at com.app.db.Connector.connect(Connector.java:45) \| at com.app.Main.start(Main.java:12) |
| 2024-11-01 | 10:00:10 | INFO | Retrying connection |
| 2024-11-01 | 10:00:15 | ERROR | Failed to connect to database \| Connection refused |
| 2024-11-01 | 10:00:20 | INFO | Connection established |

In [0]:
app_log = spark.read.text("/Workspace/Users/jeevan.busi8008@gmail.com/spark-practice/data/app_logs_multiline.txt")
app_log_with_id = app_log.withColumns(
    {
        "value": ltrim(col("value")),
        "row_id": monotonically_increasing_id()
    }
)

app_log_df = app_log_with_id.withColumn("row_identity", when(col("value").rlike("^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"), 1).otherwise(0))

window_logic = Window.orderBy(col("row_id")).rowsBetween(Window.unboundedPreceding, Window.currentRow)

app_log_recrods_df = app_log_df.withColumn("record_id", sum(col("row_identity")).over(window_logic)) \
    .groupBy("record_id").agg(
        sort_array(collect_list(struct(col("row_id"), col("value")))).alias("sorted_lines")
    ) \
    .withColumn("record", concat_ws(" | ", col("sorted_lines.value")))

app_log_recrods_df = app_log_recrods_df.select("record_id", "record")

app_log_recrods_df = app_log_recrods_df.withColumns(
    {
        "log_date": regexp_extract(col("record"), r"^(\d{4}-\d{2}-\d{2})", 1),
        "log_time": regexp_extract(col("record"), r"^\d{4}-\d{2}-\d{2} (\d{2}:\d{2}:\d{2})", 1),
        "log_level": regexp_extract(col("record"), r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} (\w+)", 1),
        "full_message": regexp_extract(col("record"), r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2} \w+ (.*)", 1)
    }
)
display(app_log_recrods_df.drop("record"))